# Icelandic GPT: Inference Demo

This notebook loads **saved weights and your own tokenizer**. It does not start training or download someone else's pretrained model. Cell outputs are intentionally empty: results will appear after you run the notebook with your checkpoint.

Before running, install the repository environment with `python -m pip install -e ".[dev,notebook]"` and select its Jupyter kernel. Set the paths to the checkpoint and `tokenizer.json` in the next cell. The same checkpoint works on both CPU and CUDA.


In [ ]:
from pathlib import Path
import json
import sys
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

CHECKPOINT = ROOT / "runs/main/best.pt"
TOKENIZER = ROOT / "artifacts/tokenizers/16384/tokenizer.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(4)

from icelandic_gpt.generate import load_bundle, generate_text, file_sha256

model, tokenizer, checkpoint = load_bundle(CHECKPOINT, TOKENIZER, DEVICE)
CHECKPOINT_SHA256 = file_sha256(CHECKPOINT)
print(f"Device: {DEVICE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Context: {model.config.block_size} tokens")
print(f"Vocabulary: {tokenizer.get_vocab_size():,}")
print(f"Tokenizer SHA-256: {checkpoint['tokenizer_sha256']}")
print("Model configuration:", checkpoint["model_config"])


## Text Continuation

This model is trained with a pretraining objective: the input provides the beginning of a text for the model to continue. It should not be treated as an assistant trained to answer questions. The `seed` makes sampling reproducible with the same software versions and device; CPU and CUDA results may differ.


In [ ]:
PROMPT = "Ísland er eyja í Norður-Atlantshafi. "
SEED = 42
TEMPERATURE = 0.8
TOP_P = 0.95
MAX_NEW_TOKENS = 128

result = generate_text(
    model, tokenizer, PROMPT, seed=SEED,
    temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
)
print(result["text"])
print(f"\nGenerated tokens: {result['generated_tokens']}; EOS: {result['stopped_on_eos']}")


## Different Prompts and Reproducible Seeds

Show all results, including unsuccessful ones. To assess readability, copy the complete text into Google Translate; translation does not replace evaluation on held-out data.


In [ ]:
prompts = [
    "Reykjavík er höfuðborg Íslands og ",
    "Vísindamenn hafa rannsakað ",
    "Börn í skólanum ",
]
examples = []
for prompt in prompts:
    for seed in (42, 43):
        example = generate_text(
            model, tokenizer, prompt, seed=seed,
            temperature=0.8, top_p=0.95, max_new_tokens=128,
        )
        examples.append(example)
        print(f"Seed {seed} | {prompt}")
        print(example["text"])
        print()


## Tokenization and Quantitative Evaluation

The tokenizer is verified by its SHA-256 hash: matching vocabulary sizes alone are insufficient. In byte-level BPE, a single character may span multiple tokens, so individual tokens can look unusual.


In [ ]:
text = "Þjóðin talar íslensku og börnin lesa bækur."
encoding = tokenizer.encode(text, add_special_tokens=False)
print("Text:", text)
print("Tokens:", encoding.tokens)
print("IDs:", encoding.ids)
assert tokenizer.decode(encoding.ids) == text
print("Tokens:", len(encoding.ids), "| UTF-8 bytes:", len(text.encode("utf-8")))


## Saved Results from the Full Test

The next cell displays actual reports, if they have been generated using the `icelandic_gpt.evaluate` commands in the README. It does not substitute example values. Perplexity is comparable when using the same tokenizer; bits per byte helps account for differences in segmentation. In the continuation benchmark, answers are balanced within groups: the context-free result is 25%.


In [ ]:
report_paths = {
    "Language modeling": ROOT / "reports/test.json",
    "Continuation benchmark": ROOT / "reports/benchmark-results.json",
}
for name, path in report_paths.items():
    print(name)
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        if report.get("checkpoint_sha256") != CHECKPOINT_SHA256:
            print("This report refers to a different checkpoint than the loaded model.")
        summary = {k: v for k, v in report.items() if k not in {"predictions", "generations", "sources"}}
        print(json.dumps(summary, ensure_ascii=False, indent=2))
    else:
        print(f"Not yet generated: {path}")
    print()


## Complete Fixed Prompt Gallery

The gallery contains 20 prompts saved in advance × 2 seeds. Its CLI saves the entire set of prompts before generating any continuations. Compare the same prompts and settings across checkpoints. The file is saved after every example, and the `complete` field distinguishes a finished run from an interrupted one.


In [ ]:
gallery_path = ROOT / "reports/gallery.json"
if gallery_path.exists():
    gallery = json.loads(gallery_path.read_text(encoding="utf-8"))
    if gallery.get("checkpoint_sha256") != CHECKPOINT_SHA256:
        print("This gallery refers to a different checkpoint than the loaded model.")
    print("Complete:", gallery["complete"])
    print("Examples:", len(gallery["generations"]), "/", gallery["expected_generations"])
    for item in gallery["generations"]:
        print(f"Document {item['document_id']} | seed {item['seed']}")
        print(item["text"])
        print()
else:
    print("Generate reports/gallery.json using the gallery command from README.")
